In [11]:
import pandas as pd
import os
from datetime import datetime
import pandas_gbq
import numpy as np


In [12]:
# =========================================================
# 1) CONFIGURAÇÕES
# ========================================================= C:\Users\Nelio\Desktop\mds_Notas\Guias\arquivos_do_dia

# Caminhos locais
PATH_RAIZ = r"C:\Users\Nelio\Desktop\mds_Notas\Guias\arquivos_do_dia"
PATH_SAIDA = r"C:\Users\Nelio\Desktop\mds_Notas\Guias"

# Data de hoje
DIA_HJ = datetime.now().strftime("%d-%m-%Y")

# BigQuery
PROJECT_ID = "infra-itaborai"
TABELA_NOTAS = "infra-itaborai.mds_notas.tb_notas"

In [13]:
# =========================================================
# 2) LEITURA E JUNÇÃO DOS ARQUIVOS .XLS
# =========================================================

arquivos_xls = [
    arq for arq in os.listdir(PATH_RAIZ)
    if arq.lower().endswith(".xls")
]

lista_df = []

for arq in arquivos_xls:
    caminho_arquivo = os.path.join(PATH_RAIZ, arq)

    df_temp = pd.read_html(
        caminho_arquivo,
        header=None,
        decimal=".",
        thousands=","
    )[1]

    lista_df.append(df_temp)

df_guias = pd.concat(lista_df, ignore_index=True)


In [14]:
# =========================================================
# 3) TRATAMENTOS (ETL)
# =========================================================

# --- Limpar nomes das colunas ---
df_guias.columns = df_guias.columns.astype(str).str.strip()


# --- Renomear colunas ---
df_guias = df_guias.rename(columns={
    "Emissor": "Cnpj_Emissor",
    "Razão Social": "Razão_Social_Emissor",
    "Prestador": "Cnpj_Prestador",
    "Razão Social.1": "Razão_Social_Prestador",
    "D.A.M.": "DAM",
    "Vlr. DAM": "Vlr_DAM"
})


# --- Garantir colunas como texto ---
colunas_texto = [
    "Cnpj Emissor",
    "Razão Social Emissor",
    "Cnpj Prestador",
    "Razão Social Prestador",
    "DAM",
    "Tipo",
    "Cancelada"
]

for col in colunas_texto:
    if col in df_guias.columns:
        df_guias[col] = (
            df_guias[col]
            .astype("string")
            .str.strip()
        )


# --- Tratar datas ---
for col in ["Vencimento", "Pagamento"]:
    if col in df_guias.columns:
        df_guias[col] = pd.to_datetime(
            df_guias[col],
            errors="coerce",
            dayfirst=True
        )


# --- Limpar CNPJ (somente dígitos) ---
if "Cnpj_Emissor" in df_guias.columns:
    df_guias["Cnpj_Emissor"] = (
        df_guias["Cnpj_Emissor"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.strip()
    )


# --- Remover duplicados por DAM ---
if "DAM" in df_guias.columns:
    df_guias = df_guias.drop_duplicates(
        subset=["DAM"],
        keep="first"
    )


# --- Criar coluna Razao_Cnpj Emissor ---
if (
    "Cnpj_Emissor" in df_guias.columns
    and "Razão_Social_Emissor" in df_guias.columns
):
    df_guias["Razao_Cnpj_Emissor"] = (
        df_guias["Cnpj_Emissor"].fillna("").astype("string")
        + " - "
        + df_guias["Razão_Social_Emissor"].fillna("").astype("string")
    )


# --- Filtrar CNPJs bloqueados ---
lista_bloqueio = {
    "07835050000138", "19411731000158", "83533217000194", "21004012000164",
    "26734407000136", "08537986000145", "97624678000187", "11670499000160",
    "12216767000131", "24532754000150", "64916243000157", "04753613000231",
    "63838133000151", "71672571000110", "85809284000114", "72026924000178",
    "15997136000195", "56373945000103", "41592516000150", "07019500000203",
    "47621331000102", "37249433000195", "11681189000141","15997136000195",
    "56373945000103","41592516000150"
}

if "Cnpj_Emissor" in df_guias.columns:
    df_guias = df_guias[
        ~df_guias["Cnpj_Emissor"].isin(lista_bloqueio)
    ]


# --- Padronizar Cancelada ---
if "Cancelada" in df_guias.columns:
    df_guias["Cancelada"] = (
        df_guias["Cancelada"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

In [17]:
df_guias.sort_values(by='Pagamento',ascending=False)

,Cnpj_Emissor,Razão_Social_Emissor,Cnpj_Prestador,Razão_Social_Prestador,Vencimento,Vlr_DAM,DAM,Pagamento,Tipo,Cancelada,Razao_Cnpj_Emissor
41334,11516008000121,W COSTA CONSTRUTORA LTDA,52.884.504/0001-15,TREVOMIX CONCRETAGEM INDUSTRIAL LTDA,2026-03-06,20723,142720,2026-03-06,Retido,NÃO,11516008000121 - W COSTA CONSTRUTORA LTDA
40127,09520056000141,CLINICA DA MULHER ITABORAI SOCIEDADE SIMPLES (...,NaN,NaN,2026-03-16,15232,142765,2026-03-05,ME,NÃO,09520056000141 - CLINICA DA MULHER ITABORAI SO...
41364,11865033000110,FUNDO MUNICIPAL DE ITABORAI,18.434.179/0001-50,J.F.PEIXE TRANSPORTES E LOCACOES LTDA-ME,2026-03-27,"2.218,10",142735,2026-03-05,Retido,NÃO,11865033000110 - FUNDO MUNICIPAL DE ITABORAI
41550,28741080000155,PREFEITURA MUNICIPAL DE ITABORAI,45.158.593/0001-57,"BR SMART SERVICOS, CONSTRUCAO E REFORMAS LTDA",2026-03-31,32360,142166,2026-03-05,Retido,NÃO,28741080000155 - PREFEITURA MUNICIPAL DE ITABORAI
41551,28741080000155,PREFEITURA MUNICIPAL DE ITABORAI,45.158.593/0001-57,"BR SMART SERVICOS, CONSTRUCAO E REFORMAS LTDA",2026-03-31,12230,142167,2026-03-05,Retido,NÃO,28741080000155 - PREFEITURA MUNICIPAL DE ITABORAI
...,...,...,...,...,...,...,...,...,...,...,...
42265,39595917000111,BRASVIP SEGURANCA PRIVADA LTDA,39.595.917/0001-11,BRASVIP SEGURANCA PRIVADA LTDA,2026-02-10,"1.640,18",141236,NaT,Direto,NÃO,39595917000111 - BRASVIP SEGURANCA PRIVADA LTDA
42266,39595917000111,BRASVIP SEGURANCA PRIVADA LTDA,39.595.917/0001-11,BRASVIP SEGURANCA PRIVADA LTDA,2026-02-10,"1.640,18",141237,NaT,Direto,NÃO,39595917000111 - BRASVIP SEGURANCA PRIVADA LTDA
42302,54758018000186,RIP SERVICOS E SEGURANCA LTDA,54.758.018/0001-86,RIP SERVICOS E SEGURANCA LTDA,2026-02-25,10159,141443,NaT,Direto,NÃO,54758018000186 - RIP SERVICOS E SEGURANCA LTDA
42314,59187570000185,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-04,4760,141304,NaT,Direto,NÃO,59187570000185 - BRASIL EXPRESS LOGISTICA LTDA


In [18]:
# =========================================================
# 4) BUSCAR NOTAS NO BIGQUERY
# =========================================================

sql_notas = f"""
SELECT
    CAST(Numero_Guia AS STRING) AS Numero_Guia
FROM `{TABELA_NOTAS}`
WHERE Numero_Guia IS NOT NULL
"""

df_notas = pandas_gbq.read_gbq(
    sql_notas,
    project_id=PROJECT_ID
)

df_notas["Numero_Guia"] = (
    df_notas["Numero_Guia"]
    .astype("string")
    .str.strip()
    .replace({"0": pd.NA, "": pd.NA})
)



Downloading: 100%|██████████|


In [19]:
# =========================================================
# 5) CRUZAMENTO + REGRAS DE STATUS (com np.select)
# =========================================================

# Padronizar DAM antes do cruzamento
df_guias["DAM"] = (
    df_guias["DAM"]
    .astype("string")
    .str.strip()
)

# Cruzamento
df_guias["existe_nota"] = df_guias["DAM"].isin(
    df_notas["Numero_Guia"]
)

# Data base (hoje sem horário)
hoje = pd.Timestamp.today().normalize()

# Regras com prioridade (ordem importa!)
condicoes = [
    df_guias["Cancelada"] == "SIM",
    (~df_guias["existe_nota"]) & (df_guias["Pagamento"].notna()),
    df_guias["Pagamento"].notna(),
    (~df_guias["existe_nota"]) & (df_guias["Vencimento"] < hoje),
    (~df_guias["existe_nota"]) & (df_guias["Vencimento"] >= hoje),
    (df_guias["Vencimento"].notna()) & (df_guias["Pagamento"].isna()) & (df_guias["Vencimento"] < hoje),
    (df_guias["Vencimento"].notna()) & (df_guias["Pagamento"].isna()) & (df_guias["Vencimento"] >= hoje),
]

valores = [
    "Guia Cancelada",
    "Guia Avulsa Paga",
    "Guia PAGA",
    "Guia Avulsa Vencida",
    "Guia Avulsa a Vencer",
    "Guia Vencida",
    "Guia a Vencer",
]

df_guias["sttsGuia"] = np.select(
    condicoes,
    valores,
    default="OUTRO STATUS"
)

condicoes_pagador = [
    df_guias["Tipo"] == "ME",
    df_guias["Tipo"] == "Retido",
    df_guias["Tipo"] == "Direto"
]

valores_pagador = [
    "Prestador",
    "Tomador",
    "Empresa de Fora"
]

df_guias["Quem_Paga"] = np.select(
    condicoes_pagador,
    valores_pagador,
    default="Outro"
)


In [20]:
df_guias

,Cnpj_Emissor,Razão_Social_Emissor,Cnpj_Prestador,Razão_Social_Prestador,Vencimento,Vlr_DAM,DAM,Pagamento,Tipo,Cancelada,Razao_Cnpj_Emissor,existe_nota,sttsGuia,Quem_Paga
0,00913096000189,BABY CLINICA LTDA,NaN,NaN,2022-09-15,2943,104775,2022-09-15,ME,NÃO,00913096000189 - BABY CLINICA LTDA,False,Guia Avulsa Paga,Prestador
1,47960950139987,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,877,104487,NaT,ME,SIM,47960950139987 - MAGAZINE LUIZA S/A,False,Guia Cancelada,Prestador
2,47960950139987,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,065,104488,2022-08-29,ME,NÃO,47960950139987 - MAGAZINE LUIZA S/A,False,Guia Avulsa Paga,Prestador
3,09492725000119,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,73600,104555,NaT,ME,NÃO,09492725000119 - FIRE SERVICE DO BRASIL LTDA ME,False,Guia Avulsa Vencida,Prestador
4,09492725000119,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,46500,104556,NaT,ME,NÃO,09492725000119 - FIRE SERVICE DO BRASIL LTDA ME,False,Guia Avulsa Vencida,Prestador
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42313,59187570000185,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-16,4510,141306,2026-02-18,Direto,NÃO,59187570000185 - BRASIL EXPRESS LOGISTICA LTDA,False,Guia Avulsa Paga,Empresa de Fora
42314,59187570000185,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-04,4760,141304,NaT,Direto,NÃO,59187570000185 - BRASIL EXPRESS LOGISTICA LTDA,False,Guia Avulsa Vencida,Empresa de Fora
42315,62294600000167,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,62.294.600/0001-67,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,2026-01-29,35860,141199,NaT,Direto,NÃO,62294600000167 - JACKELINE ARAGAO - PSICOLOGIA...,False,Guia Avulsa Vencida,Empresa de Fora
42316,72934748000172,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,72.934.748/0001-72,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,2026-02-16,"1.830,35",141672,2026-02-13,Direto,NÃO,72934748000172 - MAGIC GAMES EMPREENDIMENTOS C...,False,Guia Avulsa Paga,Empresa de Fora


In [ ]:
# # =========================================================
# # 6) SUBIR PARA O BIGQUERY
# # =========================================================

# TABELA_DESTINO = "infra-itaborai.mds_notas.fGuias"

# pandas_gbq.to_gbq(
#     df_guias,
#     destination_table=TABELA_DESTINO,
#     project_id=PROJECT_ID,
#     if_exists="replace"
# )

# print("Upload para BigQuery concluído!")
# print(f"Linhas enviadas: {len(df_guias)}")

100%|██████████| 1/1 [00:00<00:00, 2548.18it/s]

Upload para BigQuery concluído!
Linhas enviadas: 41602
